## Laboratorio 4

- Diego Valenzuela 22309 
- Daniel Dubon 22233
- Nelson García Bravatti 22434
- Joaquin Puente 22296

### Task 1 Diseño

1. Espacio de Estados y Espacio de Acciones

    Espacio de Estados: El almacen es una cuadricula de 8 x 8.

    Eleccioó: gymnasium spaces Discrete 64

    Justificación: Un espacio Discreto 64 mapea cada celda bidimensional a un unico numero entero entre 0 y 63 y esto es ideal y necesario porque tanto SARSA como Q Learning, en sus formas clasicas tabulares, requieren un estado discreto para usarse como indice en la matriz o Q Table de dimensiones 64 x 4.

    Por que no usar Box: La clase Box se utiliza para representar espacios continuos como arreglos de numeros decimales. Si se usara Box el espacio de estados seria infinito y no se podria construir una Q Table clasica y nos veriamos obligados a usar aproximadores de funciones lo cual esta fuera del alcance de una comparacion estandar de SARSA contra Q Learning.

    Espacio de Acciones: Movimientos direccionales del robot.

    Elección: gymnasium spaces Discrete 4

    Justificación: Representa 4 acciones deterministas y mutuamente excluyentes: Arriba 0, Abajo 1, Izquierda 2, Derecha 3.


2. Función de Recompensa

    - Recompensa por entrega exitosa: +100
    - Penalización por zona de congestion: -20 y se aplica cada vez que se pisa o permanece en una celda de congestion.
    - Penalización por paso:  -1 y se aplica en cualquier otra celda transitable.


    Justificación de las magnitudes:

    El objetivo es que el robot llegue a la meta de la forma mas rapida por la penalizacion por paso y segura por la penalización por congestion y aumentar 100 garantiza que llegar a la meta sea el evento dominante y el objetivo global del episodio, El menos 1 fuerza al robot a buscar la ruta mas corta y el menos 20 es lo suficientemente grande como para disuadir al robot de cruzar la congestion tambien cabe resaltar que cruzar una celda de congestion cuesta lo mismo que dar 20 pasos normales.

    Comportamientos indeseables por mala ponderación:

    Si la penalización por congestion es menor o igual a la penalización por paso el robot desarrollara un comportamiento temerario y si rodear la congestion toma 3 pasos extra pero cruzarla cuesta menos puntos el agente preferira atravesar la congestion. Si la penalizacion por paso es cero o positiva el robot perdera el sentido de urgencia y caminara en circulos infinitamente sin incentivo para llegar a la meta. Si la penalización por congestion es excesivamente alta, por ejemplo menos 1000: Podria provocar que durante la exploracion temprana el agente asocie todo el entorno con un peligro extremo y aprenda una politica suboptima de quedarse quieto o chocar contra la pared constantemente para terminar el episodio rapidamente y evitar castigos.

3. Condición de Terminación del Episodio

    El episodio terminara por cualquiera de estas dos condiciones:

    - Terminated o Estado Terminal Natural: El robot alcanza el estado del Punto de Entrega.
    - Truncated o Limite Maximo de Pasos: El episodio alcanza un limite, por ejemplo un maximo de 100 pasos.

    Es apropiado tener un limite maximo de pasos:

    Si, es necesario porque durante las primeras fases de entrenamiento la politica del robot es casi completamente aleatoria y sin un limite de pasos el robot podria entrar en bucles infinitos en zonas sin penalizaciones, o simplemente vagar sin encontrar la meta jamas y el limite asegura que los episodios finalicen, garantizando que el agente experimente repetidas veces el reinicio del entorno y converja matematicamente.

4. Diseño del Mapa 8 por 8

    Para que la diferencia entre una politica agresiva y una politica conservadora sea empiricamente visible diseñamos el mapa inspirandonos en el problema clasico de Cliff Walking.

    Leyenda de las celdas:
    S: Punto de recogida
    G: Punto de entrega
    P: Pasillo transitable
    O: Obstaculo fijo
    C: Zona de congestion

    Esquema de las primeras 4 filas:

    --

    S P P P P P P G

    O C C C C C C P

    P P P P P P P P

    P O O P O O O P

    --

    Analisis de la configuración espacial:

    En este mapa la primera fila representa una ruta directa muy rapida entre el punto de recogida y la meta, sin embargo justo debajo hay una hilera de Zonas de Congestion y si un robot esta en la fila de arriba y toma una decision aleatoria hacia abajo caera en la congestion, existe una segunda ruta segura por debajo de los obstaculos pero toma muchos mas pasos.

    Comportamiento diferencial esperado:

    Con Q Learning Agresivo y Off Policy esperamos que aprenda la funcion de valor optima asumiendo que actuara siempre de forma ambiciosa, en este caso aprendera que la ruta optima es ir en linea recta por la fila de arriba, sin embargo durante el entrenamiento con mucha exploración, su constante curiosidad hara que resbale frecuentemente hacia abajo sufriendo muchas penalizaciones y su politica es optima en la teoria pero altamente riesgosa en la practica.

    Con SARSA conservador y On Policy: Aqui se actualizan sus valores basandose en la accion real que va a tomar incluyendo los movimientos aleatorios de exploracion donde SARSA aprende que caminar por la fila de arriba es peligroso porque hay probabilidad de terminar en la congestión, por lo tanto SARSA convergera hacia una ruta mas larga pero segura, bajando para alejarse de la zona de congestión, minimizando las penalizaciones durante el entrenamiento a costa de dar mas pasos.

### Preguntas previas a la implementación

**1. Predicción formal: recompensa acumulada en entrenamiento vs. evaluación greedy**

Las reglas de actualización son:

- SARSA (on-policy): $Q(s,a) \leftarrow Q(s,a) + \alpha\left[r + \gamma Q(s',a') - Q(s,a)\right]$, donde $a'$ es la acción realmente elegida por la política $\varepsilon$-greedy en $s'$.
- Q-Learning (off-policy): $Q(s,a) \leftarrow Q(s,a) + \alpha\left[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\right]$, sin importar qué acción se ejecute realmente.

En nuestro mapa, la fila 0 (ruta corta S→G) es adyacente a una fila completa de congestión (fila 1). Para cualquier estado de la fila 0, la acción "abajo" cae en congestión (-20).

- Durante **entrenamiento**, SARSA incorpora en su target el valor esperado bajo la política que realmente se ejecuta, es decir, con probabilidad $\varepsilon$ el agente puede tomar una acción aleatoria y caer en congestión. Esto hace que $Q(s,\text{arriba/derecha})$ de los estados de la fila 0 quede penalizado por ese riesgo, empujando a SARSA a preferir la ruta segura de la fila 2 (más pasos, pero sin exposición a -20). Resultado: **SARSA acumula mayor recompensa durante el entrenamiento** porque su política de comportamiento evita activamente la congestión mientras explora.
- Q-Learning usa $\max_{a'}Q(s',a')$ en el target, ignorando que la acción real puede ser exploratoria. Por lo tanto, aprende que la fila 0 es óptima (siempre y cuando actúe greedy), pero mientras entrena con $\varepsilon$-greedy sigue tomando pasos aleatorios que lo tiran a la congestión con frecuencia. Resultado: **Q-Learning tiene menor recompensa acumulada durante el entrenamiento** (más caídas en zona de congestión, análogo al problema clásico de "Cliff Walking").
- Durante **evaluación con política greedy pura** ($\varepsilon=0$), ya no hay exploración: el entorno es determinista, así que no hay riesgo de "resbalar". Q-Learning ejecuta la ruta óptima real (fila 0, distancia mínima = 7 pasos), obteniendo la recompensa más alta posible. SARSA, en cambio, quedó convergido a la ruta conservadora de la fila 2 (más larga), por lo que su recompensa en evaluación greedy es menor que la de Q-Learning aunque siga siendo positiva y estable.

En resumen: **SARSA gana en entrenamiento, Q-Learning gana en evaluación greedy**. Esto es exactamente el resultado esperado por la propiedad on-policy vs. off-policy combinada con la geometría tipo "cliff" del mapa.

**2. Efecto de $\varepsilon$ y condición de convergencia a políticas idénticas**

El target de SARSA puede escribirse como una esperanza sobre la política de comportamiento:

$$\mathbb{E}_{a'\sim\pi_\varepsilon}[Q(s',a')] = (1-\varepsilon)\max_{a'}Q(s',a') + \frac{\varepsilon}{|A|}\sum_{a'}Q(s',a')$$

mientras que el target de Q-Learning es directamente $\max_{a'}Q(s',a')$.

La diferencia entre ambos targets es:

$$\Delta = \max_{a'}Q(s',a') - \mathbb{E}_{a'\sim\pi_\varepsilon}[Q(s',a')] = \varepsilon\left(\max_{a'}Q(s',a') - \frac{1}{|A|}\sum_{a'}Q(s',a')\right)$$

Esta expresión es **lineal en $\varepsilon$** y su segundo factor es siempre $\geq 0$ (el máximo nunca es menor que el promedio). Por lo tanto:

- Cuanto mayor es $\varepsilon$, mayor es la brecha entre SARSA y Q-Learning, y más conservadora se vuelve la política de SARSA respecto a la fila de congestión.
- Cuando $\varepsilon \to 0$, $\Delta \to 0$: las dos reglas de actualización se vuelven **matemáticamente idénticas** ($\pi_\varepsilon \to$ greedy), por lo que ambos algoritmos convergen a la misma política óptima determinista $\pi^*$ (la ruta por la fila 0).

Sí existe entonces un valor de $\varepsilon$ para el cual ambos convergen a políticas idénticas: **$\varepsilon = 0$** (equivalente a que $\varepsilon$ decaiga a 0 durante el entrenamiento, cumpliendo condiciones GLIE). La salvedad es que con $\varepsilon=0$ fijo desde el inicio no hay exploración, por lo que la garantía de convergencia al óptimo requiere que $\varepsilon$ decaiga gradualmente (arrancando $>0$ para explorar y tendiendo a 0), no que sea 0 desde el primer episodio.

**3. Cota superior de $V^*(s_0)$**

Asumiendo que el agente sigue siempre la ruta más corta sin pisar congestión, la ruta óptima es la fila 0 completa: $S=(0,0) \to G=(0,7)$, con distancia Manhattan $d = 7$ pasos (sin obstáculos ni congestión en esa fila).

De esos 7 pasos, los primeros $d-1=6$ caen en celdas de pasillo (-1 cada uno) y el último paso entra a la meta (+100). Con factor de descuento $\gamma$, la cota superior es:

$$V^*(s_0) \;\leq\; \sum_{t=0}^{d-2} \gamma^{t}\,(-1) \;+\; \gamma^{\,d-1}\,(100) \;=\; -\left(\frac{1-\gamma^{6}}{1-\gamma}\right) + 100\,\gamma^{6}$$

Casos concretos:

- Sin descuento ($\gamma=1$): $V^*(s_0) \leq 6(-1) + 100 = 94$.
- Con $\gamma=0.99$: $V^*(s_0) \leq -5.853 + 100(0.941) \approx 88.3$.
- Con $\gamma=0.9$: $V^*(s_0) \leq -4.686 + 100(0.531) \approx 48.4$.

Esta cota (94 para el caso no descontado, que es el que usamos en la implementación tabular) sirve como referencia: cualquier retorno de evaluación greedy que obtengamos de SARSA o Q-Learning debe estar por debajo o igual a 94, y qué tan cerca esté cada algoritmo de ese valor mide su cercanía al óptimo teórico.
